# 06 - Dimensionality Audit

This notebook evaluates PCA and UMAP as scientific audit tools for the WR detector. It does not train production models, modify datasets, or synchronize results to DuckDB.

Objective: test whether the current or enriched feature space shows useful structure for reviewing known WR stars, false positives, model top-K candidates, and calibration negatives.

## How To Read This Notebook

- `training_history.duckdb` provides `run_id`, `result_id`, metrics, and stored predictions for `train_oof` and `holdout`.
- Reduced parquet files under `data/processed/modeling/` provide features and splits (`train`, `holdout`, `threshold_calibration`).
- The main join uses `source_id`; any score-based visualization is tied to the selected `result_id`.
- PCA and UMAP are visual diagnostics here. They do not replace physical features and do not change the operational ranking.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import warnings

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import duckdb
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import plotly.express as px
except ImportError:
    px = None

try:
    import umap.umap_ as umap
except ImportError:
    umap = None

from wr_detector.config import load_yaml, resolve_path
from wr_detector.modeling.explorer import load_run_results, rank_models
from wr_detector.modeling.negative_reduction import negative_ratio_label, reduced_dataset_path
from wr_detector.modeling.second_layer import add_derived_features

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
warnings.filterwarnings("ignore", category=FutureWarning)

MODELS_CONFIG_PATH = ROOT / "configs" / "models.yaml"
SECOND_LAYER_CONFIG_PATH = ROOT / "configs" / "second_layer.yaml"
models_config = load_yaml(MODELS_CONFIG_PATH)
second_layer_config = load_yaml(SECOND_LAYER_CONFIG_PATH)
history_db = resolve_path(models_config["outputs"]["training_history_db"])

print(f"ROOT: {ROOT}")
print(f"Training history DB: {history_db}")

## Parameters

If `RESULT_ID = None`, the notebook automatically selects the best result in `TRAINING_RUN_ID` according to `RANK_METRIC`, with tie-breakers defined by `wr_detector.modeling.explorer.rank_models`.

In [ ]:
TRAINING_RUN_ID = "run_202606_science_v2"
RESULT_ID = None  # Example: "416393a59bb298ab3270980cf93fc257db6f7f68"

RANK_METRIC = "holdout_wr_at_100"
FEATURE_SPACE = "enriched_tabular"  # "current_colors", "colors_parallax", "colors_parallax_error", "enriched_tabular"

TOP_K = 100
MAX_NEGATIVE_POINTS = 6000
RANDOM_STATE = 42

RUN_UMAP = True
UMAP_N_NEIGHBORS = 25
UMAP_MIN_DIST = 0.08

# This can be slow because it scores the threshold_calibration pool.
SCORE_THRESHOLD_CALIBRATION = False
MAX_THRESHOLD_CALIBRATION_POINTS = 8000

## Runs And Selected Model

In [ ]:
if not history_db.exists():
    raise FileNotFoundError(f"Missing training_history.duckdb: {history_db}")

with duckdb.connect(str(history_db), read_only=True) as con:
    runs = con.execute(
        """
        SELECT run_id, run_name, imported_at, row_count, source_csv
        FROM training_runs
        ORDER BY imported_at DESC
        """
    ).fetchdf()

display(runs)

results = load_run_results(MODELS_CONFIG_PATH, run_id=TRAINING_RUN_ID)
if results.empty:
    raise ValueError(f"No results found for TRAINING_RUN_ID={TRAINING_RUN_ID!r}")

ranked = rank_models(results, metric=RANK_METRIC)
display_columns = [
    "run_id", "result_id", "dataset_variant", "feature_set", "model", "sampler",
    "negative_ratio_label", "wr_holdout", "holdout_wr_at_50", "holdout_wr_at_100",
    "holdout_average_precision", "holdout_recall_at_fpr_0p005", "overfit_warning_flag",
]
display(ranked[[c for c in display_columns if c in ranked.columns]].head(12))

if RESULT_ID is None:
    selected = ranked.iloc[0]
else:
    match = results[results["result_id"].astype(str).eq(str(RESULT_ID))]
    if match.empty:
        raise ValueError(f"RESULT_ID={RESULT_ID!r} does not belong to TRAINING_RUN_ID={TRAINING_RUN_ID!r}")
    selected = match.iloc[0]

selected = selected.copy()
display(selected[[c for c in display_columns if c in selected.index]].to_frame("selected_value"))

## Reduced Dataset And Synchronized Predictions

The parquet path is resolved from the selected model `dataset_variant` and `negative_ratio`. For historical runs without `negative_ratio_label`, the default `10x` ratio is used.

In [ ]:
def selected_negative_ratio(row: pd.Series) -> int | str:
    value = row.get("negative_ratio", np.nan)
    if pd.notna(value):
        return int(value)
    label = row.get("negative_ratio_label", np.nan)
    if pd.isna(label) or str(label).strip() == "":
        return int(models_config["negative_reduction"]["negative_to_wr_ratio"])
    label = str(label)
    if label == "all_train_negatives":
        return "all"
    if label.endswith("x"):
        return int(label[:-1])
    return int(label)

variant = str(selected["dataset_variant"])
negative_ratio = selected_negative_ratio(selected)
dataset_path = reduced_dataset_path(models_config, variant, negative_ratio=negative_ratio)
if not dataset_path.exists():
    raise FileNotFoundError(f"Missing reduced parquet: {dataset_path}")

dataset = pd.read_parquet(dataset_path)
required_columns = {"source_id", "target", "modeling_split"}
missing_required = sorted(required_columns - set(dataset.columns))
if missing_required:
    raise KeyError(f"Missing required columns in {dataset_path}: {missing_required}")

with duckdb.connect(str(history_db), read_only=True) as con:
    predictions = con.execute(
        """
        SELECT source_id, split AS prediction_split, target AS prediction_target, score, predicted, threshold
        FROM model_predictions
        WHERE run_id = ? AND result_id = ?
        """,
        [str(selected["run_id"]), str(selected["result_id"])],
    ).fetchdf()

summary = pd.crosstab(dataset["modeling_split"], dataset["target"], margins=True)
display(summary)
print(f"Dataset: {dataset_path}")
print(f"Rows dataset={len(dataset):,}; prediction rows={len(predictions):,}")

## Feature space

`current_colors` uses the exact selected model feature set. `enriched_tabular` uses the second-layer feature list and computes derived features with the same pipeline function.

In [ ]:
def requested_features(feature_space: str, selected_row: pd.Series) -> list[str]:
    if feature_space == "current_colors":
        return list(models_config["feature_sets"][str(selected_row["feature_set"])])
    if feature_space in models_config["feature_sets"]:
        return list(models_config["feature_sets"][feature_space])
    if feature_space in second_layer_config["feature_sets"]:
        return list(second_layer_config["feature_sets"][feature_space])
    raise KeyError(f"Unrecognized feature space: {feature_space}")

working = add_derived_features(dataset)
feature_candidates = requested_features(FEATURE_SPACE, selected)
available_features = [c for c in feature_candidates if c in working.columns and not working[c].isna().all()]
missing_features = [c for c in feature_candidates if c not in available_features]
if len(available_features) < 2:
    raise ValueError(f"Not enough available features for {FEATURE_SPACE}: {available_features}")

feature_audit = pd.DataFrame(
    {
        "feature": feature_candidates,
        "available": [c in available_features for c in feature_candidates],
        "non_null_pct": [float(working[c].notna().mean() * 100) if c in working.columns else np.nan for c in feature_candidates],
    }
)
display(feature_audit)
print(f"Available features: {len(available_features)} / {len(feature_candidates)}")
if missing_features:
    print("Omitted features:", missing_features)

## Visual Sample And Labels

The sample keeps all WR rows, all top-K sources for the selected model, and a reproducible negative sample to avoid saturating plots.

In [ ]:
predictions = predictions.drop_duplicates("source_id")
plot_df = working.merge(predictions, on="source_id", how="left")
plot_df["score"] = pd.to_numeric(plot_df["score"], errors="coerce")
plot_df["predicted"] = pd.to_numeric(plot_df["predicted"], errors="coerce")
plot_df["rank_within_prediction_split"] = plot_df.groupby("prediction_split")["score"].rank(method="first", ascending=False)
plot_df["top_k"] = plot_df["rank_within_prediction_split"].le(TOP_K)
plot_df["false_positive"] = plot_df["target"].eq(0) & plot_df["predicted"].eq(1)
plot_df["false_negative"] = plot_df["target"].eq(1) & plot_df["predicted"].eq(0)
plot_df["class_label"] = np.where(plot_df["target"].eq(1), "WR", "non-WR")
plot_df["review_label"] = "context negative"
plot_df.loc[plot_df["target"].eq(1), "review_label"] = "known WR"
plot_df.loc[plot_df["top_k"].fillna(False) & plot_df["target"].eq(0), "review_label"] = "top-K negative"
plot_df.loc[plot_df["false_positive"].fillna(False), "review_label"] = "threshold false positive"
plot_df.loc[plot_df["false_negative"].fillna(False), "review_label"] = "threshold false negative"

rng = np.random.default_rng(RANDOM_STATE)
must_keep = plot_df["target"].eq(1) | plot_df["top_k"].fillna(False) | plot_df["false_positive"].fillna(False) | plot_df["false_negative"].fillna(False)
negative_pool = plot_df[~must_keep & plot_df["target"].eq(0)]
negative_sample_index = rng.choice(
    negative_pool.index.to_numpy(),
    size=min(MAX_NEGATIVE_POINTS, len(negative_pool)),
    replace=False,
) if len(negative_pool) else []
visual_df = pd.concat([plot_df.loc[must_keep], plot_df.loc[negative_sample_index]], ignore_index=False).copy()
visual_df = visual_df.sort_values(["target", "score"], ascending=[True, True], na_position="first").reset_index(drop=True)

display(visual_df["review_label"].value_counts(dropna=False).rename_axis("review_label").reset_index(name="rows"))
print(f"Visual rows={len(visual_df):,}; all dataset rows={len(plot_df):,}")

## PCA: Explained Variance And Physical Interpretation

In [ ]:
X_all = working[available_features].replace([np.inf, -np.inf], np.nan)
pca_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(random_state=RANDOM_STATE)),
    ]
)
pca_pipeline.fit(X_all)
pca_model = pca_pipeline.named_steps["pca"]
explained = pd.DataFrame(
    {
        "component": [f"PC{i+1}" for i in range(len(pca_model.explained_variance_ratio_))],
        "explained_variance_ratio": pca_model.explained_variance_ratio_,
        "cumulative_variance": np.cumsum(pca_model.explained_variance_ratio_),
    }
)
thresholds = [0.80, 0.90, 0.95, 0.99]
variance_summary = pd.DataFrame(
    {
        "variance_threshold": thresholds,
        "n_components": [int(np.searchsorted(explained["cumulative_variance"], t) + 1) for t in thresholds],
    }
)
display(variance_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=explained.head(15), x="component", y="explained_variance_ratio", color="#4C78A8", ax=axes[0])
axes[0].set_title("PCA explained variance")
axes[0].set_xlabel("")
axes[0].set_ylabel("Variance ratio")
axes[0].tick_params(axis="x", rotation=45)
sns.lineplot(data=explained.head(20), x="component", y="cumulative_variance", marker="o", color="#D95F02", ax=axes[1])
axes[1].axhline(0.95, color="#333333", linestyle="--", linewidth=1, label="95%")
axes[1].set_ylim(0, 1.02)
axes[1].set_title("Cumulative variance")
axes[1].set_xlabel("")
axes[1].set_ylabel("Cumulative variance")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()
plt.tight_layout()
plt.show()

loadings = pd.DataFrame(
    pca_model.components_.T,
    index=available_features,
    columns=[f"PC{i+1}" for i in range(len(available_features))],
)
top_loading_rows = []
for pc in ["PC1", "PC2", "PC3", "PC4"]:
    if pc in loadings.columns:
        part = loadings[pc].rename("loading").reset_index().rename(columns={"index": "feature"})
        part["component"] = pc
        part["abs_loading"] = part["loading"].abs()
        top_loading_rows.append(part.sort_values("abs_loading", ascending=False).head(8))
top_loadings = pd.concat(top_loading_rows, ignore_index=True) if top_loading_rows else pd.DataFrame()
display(top_loadings[["component", "feature", "loading", "abs_loading"]])

## PCA 2D: WR, Top-K, And False Positives

In [ ]:
pca_coords = pca_pipeline.transform(visual_df[available_features].replace([np.inf, -np.inf], np.nan))[:, :2]
visual_pca = visual_df.copy()
visual_pca["PC1"] = pca_coords[:, 0]
visual_pca["PC2"] = pca_coords[:, 1]

hover_cols = [c for c in ["source_id", "target", "modeling_split", "prediction_split", "score", "rank_within_prediction_split", "simbad_main_type"] if c in visual_pca.columns]

if px is not None:
    fig = px.scatter(
        visual_pca,
        x="PC1",
        y="PC2",
        color="review_label",
        symbol="class_label",
        hover_data=hover_cols,
        color_discrete_map={
            "context negative": "#9AA0A6",
            "known WR": "#D4A017",
            "top-K negative": "#CC6677",
            "threshold false positive": "#B2182B",
            "threshold false negative": "#2166AC",
        },
        title=f"PCA audit - {variant} / {FEATURE_SPACE} / top-{TOP_K}",
        width=980,
        height=680,
    )
    fig.update_traces(marker={"size": 7, "opacity": 0.72, "line": {"width": 0.3, "color": "white"}})
    fig.update_layout(legend_title_text="Review label")
    fig.show()
else:
    plt.figure(figsize=(9, 7))
    sns.scatterplot(data=visual_pca, x="PC1", y="PC2", hue="review_label", style="class_label", alpha=0.7, s=36)
    plt.title(f"PCA audit - {variant} / {FEATURE_SPACE}")
    plt.tight_layout()
    plt.show()

## UMAP 2D: Exploratory Local Structure

UMAP is visual only. Its coordinates should not be treated as direct physical evidence or as operational features without separate validation.

In [ ]:
if not RUN_UMAP:
    print("RUN_UMAP=False; UMAP is skipped.")
elif umap is None:
    print("umap-learn is not installed. Install it only for this exploratory visualization.")
else:
    umap_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "umap",
                umap.UMAP(
                    n_components=2,
                    n_neighbors=UMAP_N_NEIGHBORS,
                    min_dist=UMAP_MIN_DIST,
                    metric="euclidean",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )
    umap_coords = umap_pipeline.fit_transform(visual_df[available_features].replace([np.inf, -np.inf], np.nan))
    visual_umap = visual_df.copy()
    visual_umap["UMAP1"] = umap_coords[:, 0]
    visual_umap["UMAP2"] = umap_coords[:, 1]
    if px is not None:
        fig = px.scatter(
            visual_umap,
            x="UMAP1",
            y="UMAP2",
            color="review_label",
            symbol="class_label",
            hover_data=hover_cols,
            color_discrete_map={
                "context negative": "#9AA0A6",
                "known WR": "#D4A017",
                "top-K negative": "#CC6677",
                "threshold false positive": "#B2182B",
                "threshold false negative": "#2166AC",
            },
            title=f"UMAP audit - {variant} / {FEATURE_SPACE} / top-{TOP_K}",
            width=980,
            height=680,
        )
        fig.update_traces(marker={"size": 7, "opacity": 0.72, "line": {"width": 0.3, "color": "white"}})
        fig.update_layout(legend_title_text="Review label")
        fig.show()
    else:
        plt.figure(figsize=(9, 7))
        sns.scatterplot(data=visual_umap, x="UMAP1", y="UMAP2", hue="review_label", style="class_label", alpha=0.7, s=36)
        plt.title(f"UMAP audit - {variant} / {FEATURE_SPACE}")
        plt.tight_layout()
        plt.show()

## Diagnostics By Score And SIMBAD Type

These views help distinguish whether false positives form a coherent astrophysical population or dispersed noise.

In [ ]:
scored = visual_pca[visual_pca["score"].notna()].copy()
if scored.empty:
    print("No synchronized scores are available for the visualized rows.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.histplot(data=scored, x="score", hue="class_label", bins=40, common_norm=False, stat="density", ax=axes[0])
    axes[0].set_title("Score distribution by target")
    axes[0].set_xlabel("WR score")
    top_negative_types = (
        scored[scored["target"].eq(0) & scored["top_k"].fillna(False)]
        .get("simbad_main_type", pd.Series(dtype="object"))
        .fillna("missing")
        .value_counts()
        .head(12)
        .rename_axis("simbad_main_type")
        .reset_index(name="rows")
    )
    if top_negative_types.empty:
        axes[1].axis("off")
        axes[1].set_title("No top-K negatives in visual sample")
    else:
        sns.barplot(data=top_negative_types, y="simbad_main_type", x="rows", color="#CC6677", ax=axes[1])
        axes[1].set_title(f"Top-{TOP_K} negative SIMBAD types")
        axes[1].set_xlabel("Rows")
        axes[1].set_ylabel("")
    plt.tight_layout()
    plt.show()
    display(top_negative_types)

## Optional Threshold Calibration

`threshold_calibration` is not in `model_predictions` because it does not participate in CV or holdout. This cell loads the trained model and scores a sample of those negatives to audit possible false positives outside the reduced training set.

In [ ]:
if not SCORE_THRESHOLD_CALIBRATION:
    print("SCORE_THRESHOLD_CALIBRATION=False; this optional audit is skipped.")
else:
    model_path = resolve_path(str(selected["model_path"]))
    if not model_path.exists():
        raise FileNotFoundError(f"Missing trained model: {model_path}")
    model = joblib.load(model_path)
    model_features = list(models_config["feature_sets"][str(selected["feature_set"])])
    calibration = working[working["modeling_split"].eq("threshold_calibration")].copy()
    if calibration.empty:
        raise ValueError("The dataset does not contain the `threshold_calibration` split.")
    if len(calibration) > MAX_THRESHOLD_CALIBRATION_POINTS:
        calibration = calibration.sample(MAX_THRESHOLD_CALIBRATION_POINTS, random_state=RANDOM_STATE)
    x_cal = calibration[model_features].replace([np.inf, -np.inf], np.nan)
    if hasattr(model, "predict_proba"):
        calibration["score"] = model.predict_proba(x_cal)[:, 1]
    elif hasattr(model, "decision_function"):
        calibration["score"] = model.decision_function(x_cal)
    else:
        raise TypeError("The model exposes neither `predict_proba` nor `decision_function`.")
    calibration["review_label"] = "threshold calibration negative"
    calibration["class_label"] = "non-WR"
    calibration["top_k"] = calibration["score"].rank(method="first", ascending=False).le(TOP_K)
    calibration_pca = pca_pipeline.transform(calibration[available_features].replace([np.inf, -np.inf], np.nan))[:, :2]
    calibration["PC1"] = calibration_pca[:, 0]
    calibration["PC2"] = calibration_pca[:, 1]
    cal_plot = calibration[calibration["top_k"]].copy()
    context = visual_pca[visual_pca["target"].eq(1)].copy()
    cal_visual = pd.concat([context, cal_plot], ignore_index=True, sort=False)
    if px is not None:
        fig = px.scatter(
            cal_visual,
            x="PC1",
            y="PC2",
            color="review_label",
            hover_data=[c for c in ["source_id", "score", "simbad_main_type"] if c in cal_visual.columns],
            title=f"PCA audit - top-{TOP_K} threshold calibration negatives vs known WR",
            width=980,
            height=640,
        )
        fig.update_traces(marker={"size": 8, "opacity": 0.75, "line": {"width": 0.3, "color": "white"}})
        fig.show()
    display(cal_plot.sort_values("score", ascending=False).head(TOP_K)[[c for c in ["source_id", "score", "simbad_main_type"] if c in cal_plot.columns]])

## Suggested Scientific Reading

Use these questions to decide whether any view should be promoted to the app or to a formal experiment:

1. Do known WR stars form one compact region or several interpretable regions?
2. Do top-K negatives fall near the WR locus or remain dispersed?
3. Do false positives share SIMBAD types or photometric properties?
4. Does PCA preserve enough variance with few components without erasing WR/non-WR separation?
5. Does UMAP reveal additional structure beyond PCA, or mainly reorganize noise?

If questions 2-3 have clear answers, the natural next step is a read-only Model Explorer page. Otherwise, keep this as a notebook audit.